# 1: Colexification

* document important functions & steps
* think about could be sent to a .py script (tho not necessary if too much work)

In [1]:
!python -m spacy download en_core_web_sm
import dill
import json

import string
import spacy
import hunspell

import numpy as np
import pandas as pd
import scipy

import seaborn as sns

from collections import Counter
from scipy.sparse import csr_matrix

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 17.7 MB/s  0:00:009.0 MB/s eta 0:00:0101
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
spc = spacy.load('en_core_web_sm')
hsl = hunspell.HunSpell('/usr/share/hunspell/en_US.dic', '/usr/share/hunspell/en_US.aff')

In [3]:
processed_data_path = '../data/processed/'

corpus_fn = processed_data_path + 'corpus.pkl'
tes_fn = processed_data_path + 'tes.pkl'
lexicon_fn = processed_data_path + 'lexicon.json'
colex_fn = processed_data_path + 'colex.pkl'

In [4]:
SELECTED_POS = {'NOUN', 'VERB', 'ADJ'}

corpus = dill.load(open(corpus_fn, 'rb'))
tes = dill.load(open(tes_fn, 'rb'))

## Lexicon

In [5]:
### SELECTING & CLEANING LEMMAS

def clean(text):
    text = text.lower()
    text = text.encode("ascii", "ignore").decode("ascii")
    text = text.translate(str.maketrans('', '', string.punctuation))

    return text
    
def select(entry):
    if len(entry) <= 1:
        #print("remove:", entry)
        return None

    # cleaning
    entry = clean(entry)

    if not entry.isalpha():
        return None

    # for now: remove typos
    if not hsl.spell(entry):
        return None

    # for now: remove other pos
    try:   
        if spc(entry)[0].pos_ not in SELECTED_POS:
            print("wrong pos:", lemma)
            return None
    except:
        return None
        
    return entry

In [6]:
def idx2lex(lexicon, selected):
    return np.array(lexicon)[selected]

def lex2idx(lexicon, selected):
    return [lexicon.index(s) for s in selected] if type(selected)==list else lexicon.index(selected)
    
def extract_lexicon(tes):
    return sorted({select(k) for lang in tes for k in tes[lang][0].keys()} - {None})

def extract_lexicon_alt(tes):
    return sorted({clean(k) for lang in tes for k in tes[lang][0].keys()})

In [7]:
### LEXICON

lexicon = extract_lexicon(tes)
json.dump(lexicon, open(lexicon_fn, 'w'))
len(lexicon)

5970

## Colexification Matrix

In [8]:
def synlex_ratio(s1, s2, tes):
    a = {lines for stem in tes[0][s1] for lines in tes[0][s1][stem]}
    b = {lines for stem in tes[0][s2] for lines in tes[0][s2][stem]}

    return len(a.intersection(b)) / min(len(a), len(b))
    
def compute_language_colex(lexicon, tes, min_stem_freq=2, synlex_threshold=0):
    # generate builders for affinity matrix
    targets = sorted({str(t) for seed in tes[1] for stem in tes[1][seed] for t in tes[1][seed][stem]})
    row, col, data = [], [], []
    for seed in tes[1]:
        if seed not in lexicon: continue   
        for stem in tes[1][seed]:
            if tes[1][seed][stem].total() >= min_stem_freq:
                for target in tes[1][seed][stem]:
                    i = lexicon.index(seed)
                    j = targets.index(target)
                    row.append(i)
                    col.append(j)
                    data.append(1)

    # generate affinity matrix
    mx = csr_matrix((data, (row, col)), shape=(len(lexicon), len(targets)))
    clx = mx * mx.transpose()
    clx = (clx / clx.max()).ceil()
    clx.setdiag(0)
    clx.eliminate_zeros()

    # filter synlexes
    nz_rows, nz_cols = clx.nonzero()
    for r, c in list(zip(nz_rows, nz_cols)):
        if synlex_ratio(lexicon[r], lexicon[c], tes) > synlex_threshold:
            clx[r,c] = 0 
    clx.eliminate_zeros()
    
    return clx

def compute_colex(lexicon, tes,  min_stem_freq=2, synlex_threshold=0, min_clx_count=None, max_clx_size=None):
    # compute sum of colex matrices across languages
    clx = np.sum([compute_language_colex(lexicon, tes[lang], min_stem_freq, synlex_threshold) for lang in tes])
    
    # filter minimum colex count (min nlangs that colex)
    if min_clx_count:
        clx.data[clx.data < min_clx_count] = 0
        clx.eliminate_zeros()
    
    # filter maximum colex array size (max nterms in a row)
    if max_clx_size:
        larger = [i for i in set(clx.nonzero()[0]) if len(clx.getrow(i).nonzero()[1]) > max_clx_size]
        tmp = clx.toarray()
        tmp[larger, :] = 0
        tmp[:, larger] = 0

        clx = csr_matrix(tmp)
        clx.eliminate_zeros()
    
    return clx

def print_colex(lexicon, colex):
    rows = sorted(set(colex.nonzero()[0]))
    for i in rows:
        cols = colex.getrow(i).nonzero()[1]
        if len(cols) > 0:
            print(lexicon[i], ":", [(lexicon[j], int(colex[i,j])) for j in cols])
    
def generate_synlex_dataframe(lexicon, tes):
    synlex = {'lang': [], 's1': [], 's2': [], 'ratio': []}
    for lang in tes:
        clx = compute_language_colex(lexicon, tes[lang], synlex_threshold=2)
        nz_rows, nz_cols = clx.nonzero()
        for r, c in list(zip(nz_rows, nz_cols)):
            synlex['lang'].append(lang)
            synlex['s1'].append(lexicon[r])
            synlex['s2'].append(lexicon[c])
            synlex['ratio'].append(synlex_ratio(lexicon[r], lexicon[c], tes[lang]))
    return pd.DataFrame(synlex)

In [9]:
### COLEXIFICATION MATRIX

MIN_STEM_FREQ = 2
SYNLEX_THRESHOLD = 0
MIN_CLX_COUNT = 2

colex = compute_colex(lexicon, tes, MIN_STEM_FREQ, SYNLEX_THRESHOLD, MIN_CLX_COUNT)
print(colex.nnz)
dill.dump(colex, open(colex_fn, 'wb'))

748


In [ ]:
### SYNLEXIFICATIONS
synlex_df = generate_synlex_dataframe(lexicon, tes)
synlex_df.to_csv(processed_data_path + 'synlex.csv')
synlex_df

In [ ]:
sns.histplot(synlex_df.ratio)